# Suspension Telemetry Trackside Quicklook

This notebook is an experimental Colab frontend for the repo's Qt-free quicklook path. It clones the public GitHub repo and runs the same Python backend as the local quicklook script.

In [ ]:
from pathlib import Path

GITHUB_OWNER = "JoelKuula"
GITHUB_REPO = "Suspension_telemetry"
GITHUB_REF = "main"
REPO_DIR = Path("/content/Suspension_telemetry")
INPUT_PATH = "/content/drive/MyDrive/SuspensionTelemetry/LOG00010.BIN"
OUTPUT_DIR = None  # Example: "/content/drive/MyDrive/SuspensionTelemetry/quicklook_outputs/LOG00010"

## Public repo workflow

This notebook does not need GitHub credentials for the public repo. Keep private ride files in your own Drive path and point `INPUT_PATH` at the file you want to inspect.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys


repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", repo_url], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GITHUB_REF, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GITHUB_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GITHUB_REF], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_REF, repo_url, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "polars", "matplotlib"], check=True)
os.chdir(REPO_DIR)
print(f"Repo ready at {REPO_DIR}")

In [ ]:
import sys
from IPython.display import Image, Markdown, display

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from scripts.postprocess_gui_app.backend.quicklook_service import run_quicklook

result = run_quicklook(Path(INPUT_PATH), None if OUTPUT_DIR in (None, "") else Path(OUTPUT_DIR))
display(Markdown("## Quicklook Summary"))
display(Markdown(f"```text\n{result.summary_text}\n```"))
display(Markdown("## Travel Overview"))
display(Image(filename=str(result.artifact_paths["travel_overview_png"])))
display(Markdown("## Velocity Histograms"))
display(Image(filename=str(result.artifact_paths["velocity_histograms_png"])))
print(f"Quicklook outputs written to {result.output_dir}")